# Lab3.2 — AXI4-Stream și pregătirea Overlay-ului DMA

În Lab2 am utilizat AXI-Lite pentru acces la registre. Pentru volume mari de date este mai potrivit un transfer de tip **stream**, realizat cu AXI4-Stream și DMA.


## Obiective

- diferențierea AXI-Lite de AXI4-Stream;
- înțelegerea rolului DMA;
- încărcarea Overlay-ului `dma_tutorial`;
- identificarea celor două controlere DMA;
- înțelegerea traseului DDR → DMA → PL → DMA → DDR.


## 1. AXI-Lite vs AXI4-Stream

| Interfață | Utilizare tipică |
|---|---|
| **AXI-Lite** | registre de control și configurare |
| **AXI4-Stream** | flux continuu de date |

Pentru o imagine, un vector sau un flux video nu vrem să scriem fiecare element individual într-un registru din Python. DMA transferă blocuri de date între DDR și interfața AXI4-Stream fără ca procesorul să copieze fiecare element.


## 2. Designul utilizat

Overlay-ul existent în `PYNQ_Workshop` conține două DMA-uri și un AXI Stream FIFO folosit ca bloc de trecere:

```text
DDR / PS
   │
   ▼
DMA send
   │ AXI4-Stream
   ▼
AXI Stream FIFO
   │ AXI4-Stream
   ▼
DMA receive
   │
   ▼
DDR / PS
```

FIFO-ul nu modifică datele; în laboratoarele ulterioare acest loc va fi ocupat de un accelerator hardware.


## 3. Pregătirea fișierelor Overlay

Dacă fișierele nu există local, celula următoare le descarcă din repository-ul `PYNQ_Workshop`.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

BITSTREAM_DIR = Path("bitstream")
BITSTREAM_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/onigas/PYNQ_Workshop/master/Session_4/bitstream"
FILES = ("dma_tutorial.bit", "dma_tutorial.hwh")

for filename in FILES:
    local_path = BITSTREAM_DIR / filename
    if not local_path.exists():
        print(f"Downloading {filename} ...")
        try:
            urlretrieve(f"{BASE_URL}/{filename}", local_path)
        except Exception as e:
            raise RuntimeError(
                f"Nu s-a putut descărca {filename}. "
                "Copiați manual fișierele dma_tutorial.bit și dma_tutorial.hwh "
                "în directorul Lab3/bitstream."
            ) from e

print("DMA overlay files are ready.")

## 4. Încărcarea Overlay-ului


In [ ]:
from pynq import Overlay

overlay = Overlay("bitstream/dma_tutorial.bit")
print("DMA overlay loaded.")

## 5. Identificarea IP-urilor


In [ ]:
for name, info in overlay.ip_dict.items():
    print(f"{name:30s} {info.get('type', '-')}")

Selectăm IP-urile de tip AXI DMA.


In [ ]:
dma_ips = {
    name: info
    for name, info in overlay.ip_dict.items()
    if "axi_dma" in info.get("type", "").lower()
}

for name, info in dma_ips.items():
    print(f"{name:30s} base={hex(info['phys_addr'])} range={hex(info['addr_range'])}")

## 6. Obiectele DMA

Designul are un DMA pentru sensul **PS → PL** și unul pentru **PL → PS**.


In [ ]:
dma_send = overlay.axi_dma_from_ps_to_pl
dma_recv = overlay.axi_dma_from_pl_to_ps

print("Send channel available   :", hasattr(dma_send, "sendchannel"))
print("Receive channel available:", hasattr(dma_recv, "recvchannel"))

## 7. Exercițiu

Folosind `overlay.ip_dict`, afișați pentru cele două DMA-uri:

- numele IP-ului;
- tipul;
- adresa fizică;
- domeniul de adrese.


In [ ]:
# Scrieți aici soluția


## 8. Întrebări

1. De ce AXI-Lite nu este alegerea potrivită pentru transferul unei imagini mari element cu element?
2. Care este rolul DMA într-un sistem PS–PL?
3. Ce diferență există între canalul `sendchannel` și `recvchannel`?
4. Ce bloc hardware va putea înlocui FIFO-ul în laboratoarele de accelerare?
